# 05 — Single Newscast OCR + Speech Fusion

**Parte 2 — Single Video Analysis / Newscast**

Este notebook junta os resultados finais dos notebooks individuais:

- **OCR**: fornece a segmentação temporal principal, com blocos/notícias, temas visíveis no lower-third e candidatos a transição pivot/anchor → notícia/peça.
- **Speech**: fornece temas falados em janelas temporais, transcrições e evidência textual para validar/enriquecer os blocos OCR.

## Estratégia multimodal

A fusão não dá o mesmo papel às duas modalidades:

```text
OCR    → define a estrutura temporal base: blocos, fronteiras, duração, transições visuais.
Speech → valida e enriquece os temas dentro de cada bloco OCR.
```

Isto é importante porque o OCR foi desenhado para captar mudanças visuais no rodapé/lower-third, enquanto o speech foi processado em janelas temporais regulares e não tenta detetar diretamente a passagem pivot → notícia.

## Output principal

O resultado final é uma tabela por bloco OCR:

```text
block_id | start_time | end_time | duration_min
ocr_theme | speech_theme | final_theme
ocr_speech_agreement | needs_review
ocr_text | speech_excerpt
anchor_to_piece_status
```

As tabelas intermédias de diagnóstico dos notebooks OCR e speech não são repetidas aqui.


## 0. Configuração

Este notebook assume que os notebooks anteriores já foram corridos e exportaram os CSVs finais.

Entradas principais esperadas:

```text
outputs_ocr_04_single_newscast/
    ocr_consolidated_news_blocks.csv
    ocr_anchor_to_piece_candidates.csv        # opcional mas recomendado
    ocr_transition_candidates.csv             # opcional

outputs_speech_30s_topic_analysis/
    speech_30s_windows_topic_timeline.csv     # principal para fusão temporal
    speech_30s_topic_blocks_consolidated.csv  # opcional, referência speech-only
    speech_30s_transition_candidates.csv      # opcional
```


In [ ]:
from pathlib import Path
import math
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
BASE_DIR = Path(".")

# Must match the OCR and speech outputs.
# Example:
# Telejornal_RTP_Dec_2_ocr.pkl    -> Telejornal_RTP_Dec_2
# Telejornal_RTP_Dec_2_speech.pkl -> Telejornal_RTP_Dec_2
VIDEO_ID = "Telejornal_RTP_Jan_13"

OCR_OUTPUT_DIR = BASE_DIR / "outputs_ocr_by_video" / VIDEO_ID
SPEECH_OUTPUT_DIR = BASE_DIR / "outputs_speech_by_video" / VIDEO_ID
FUSION_OUTPUT_DIR = BASE_DIR / "outputs_fusion_by_video" / VIDEO_ID
FUSION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Ficheiros principais
# ------------------------------------------------------------
OCR_BLOCKS_FILE = OCR_OUTPUT_DIR / "ocr_consolidated_news_blocks.csv"
OCR_ANCHOR_FILE = OCR_OUTPUT_DIR / "ocr_anchor_to_piece_candidates.csv"
OCR_TRANSITIONS_FILE = OCR_OUTPUT_DIR / "ocr_transition_candidates.csv"

# Para a fusão, usamos janelas speech, não apenas blocos speech consolidados.
# Isto permite calcular overlap temporal ponderado com cada bloco OCR.
SPEECH_WINDOWS_FILE = SPEECH_OUTPUT_DIR / "speech_30s_windows_topic_timeline.csv"
SPEECH_BLOCKS_FILE = SPEECH_OUTPUT_DIR / "speech_30s_topic_blocks_consolidated.csv"
SPEECH_TRANSITIONS_FILE = SPEECH_OUTPUT_DIR / "speech_30s_transition_candidates.csv"

# ------------------------------------------------------------
# Parâmetros de fusão
# ------------------------------------------------------------
MIN_OVERLAP_SEC = 1
MIN_SPEECH_COVERAGE_RATIO = 0.20

# Usado para procurar candidatos speech perto do início de um bloco OCR.
SPEECH_TRANSITION_NEAR_BOUNDARY_SEC = 45

# Mostrar poucas visualizações no notebook final.
RUN_MINIMAL_PLOTS = True

print("VIDEO_ID:", VIDEO_ID)
print("OCR_OUTPUT_DIR:", OCR_OUTPUT_DIR.resolve())
print("SPEECH_OUTPUT_DIR:", SPEECH_OUTPUT_DIR.resolve())
print("FUSION_OUTPUT_DIR:", FUSION_OUTPUT_DIR.resolve())

## 1. Funções auxiliares

Estas funções fazem três coisas:

1. carregar CSVs obrigatórios/opcionais;
2. normalizar colunas de tempo;
3. comparar temas OCR vs speech.


In [ ]:
def seconds_to_hhmmss(seconds):
    if pd.isna(seconds):
        return ""
    seconds = int(round(float(seconds)))
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    if h > 0:
        return f"{h:02d}:{m:02d}:{s:02d}"
    return f"{m:02d}:{s:02d}"


def load_required_csv(path, label):
    path = Path(path)
    if not path.exists():
        available = []
        if path.parent.exists():
            available = [p.name for p in sorted(path.parent.glob("*.csv"))]
        raise FileNotFoundError(
            f"Ficheiro obrigatório não encontrado para {label}: {path}\\n"
            f"CSV disponíveis em {path.parent}: {available}\\n\\n"
            "Corre primeiro os notebooks OCR e Speech, garantindo que exportam os CSVs finais."
        )
    df = pd.read_csv(path)
    print(f"{label}: loaded {path.name} with shape {df.shape}")
    return df


def load_optional_csv(path, label):
    path = Path(path)
    if not path.exists():
        print(f"{label}: optional file not found → {path.name}")
        return None
    df = pd.read_csv(path)
    print(f"{label}: loaded {path.name} with shape {df.shape}")
    return df


def ensure_numeric_time_columns(df, start_col="start_sec", end_col="end_sec", label="df"):
    df = df.copy()

    missing = [c for c in [start_col, end_col] if c not in df.columns]
    if missing:
        raise ValueError(f"{label}: faltam colunas temporais {missing}. Colunas existentes: {list(df.columns)}")

    df[start_col] = pd.to_numeric(df[start_col], errors="coerce")
    df[end_col] = pd.to_numeric(df[end_col], errors="coerce")

    df = df.dropna(subset=[start_col, end_col]).copy()
    df[start_col] = df[start_col].astype(float)
    df[end_col] = df[end_col].astype(float)

    df["duration_sec"] = df[end_col] - df[start_col]
    df = df[df["duration_sec"] > 0].copy()

    if "start_time" not in df.columns:
        df["start_time"] = df[start_col].apply(seconds_to_hhmmss)
    if "end_time" not in df.columns:
        df["end_time"] = df[end_col].apply(seconds_to_hhmmss)
    if "duration_min" not in df.columns:
        df["duration_min"] = df["duration_sec"] / 60

    return df


def first_existing_col(df, candidates, required=False, label="df"):
    for col in candidates:
        if col in df.columns:
            return col
    if required:
        raise ValueError(f"{label}: nenhuma das colunas existe: {candidates}. Colunas disponíveis: {list(df.columns)}")
    return None


def split_items(value):
    if pd.isna(value) or str(value).strip() == "":
        return set()
    return {item.strip() for item in str(value).split(",") if item.strip()}


def concat_unique(values, max_items=20, sep=", "):
    out = []
    for v in values:
        if pd.isna(v) or str(v).strip() == "":
            continue
        parts = [p.strip() for p in str(v).split(",") if p.strip()]
        if not parts:
            parts = [str(v).strip()]
        for p in parts:
            if p not in out:
                out.append(p)
            if len(out) >= max_items:
                return sep.join(out)
    return sep.join(out)


def concat_texts(values, max_chars=1200, sep=" "):
    out = []
    seen = set()
    for v in values:
        if pd.isna(v):
            continue
        s = str(v).strip()
        if not s:
            continue
        if s not in seen:
            out.append(s)
            seen.add(s)
    text = sep.join(out)
    if len(text) > max_chars:
        return text[:max_chars] + "..."
    return text


# Grupos temáticos usados para classificar concordância média.
# Inclui nomes usados tanto no OCR como no Speech.
RELATED_THEME_GROUPS = {
    "politics": {
        "Eleições/Campanha", "Governo/Partidos", "Sondagens"
    },
    "public_services": {
        "Saúde", "Educação", "Habitação"
    },
    "economy_work": {
        "Economia", "Greves/Trabalho"
    },
    "security_justice": {
        "Justiça/Segurança", "Incêndios/Proteção Civil"
    },
    "international": {
        "Internacional"
    },
    "sports": {
        "Desporto"
    },
    "culture": {
        "Cultura"
    },
    "environment_weather": {
        "Ambiente", "Ambiente/Clima", "Meteorologia", "Incêndios/Proteção Civil"
    },
    "transport": {
        "Transportes", "Transportes/Mobilidade"
    },
}


def theme_group(theme):
    if pd.isna(theme) or str(theme).strip() in ["", "Other/Unknown", "No speech"]:
        return "unknown"
    theme = str(theme).strip()
    for group_name, themes in RELATED_THEME_GROUPS.items():
        if theme in themes:
            return group_name
    return theme


def themes_are_related(theme_a, theme_b):
    if pd.isna(theme_a) or pd.isna(theme_b):
        return False
    theme_a = str(theme_a).strip()
    theme_b = str(theme_b).strip()

    if theme_a in ["", "Other/Unknown", "No speech"] or theme_b in ["", "Other/Unknown", "No speech"]:
        return False

    return theme_group(theme_a) == theme_group(theme_b)


def agreement_between_themes(ocr_theme, speech_theme):
    ocr_theme = "Other/Unknown" if pd.isna(ocr_theme) or str(ocr_theme).strip() == "" else str(ocr_theme).strip()
    speech_theme = "No speech" if pd.isna(speech_theme) or str(speech_theme).strip() == "" else str(speech_theme).strip()

    if speech_theme == "No speech":
        return {
            "agreement": "no_speech",
            "final_theme": ocr_theme,
            "needs_review": True,
            "review_reason": "No speech window overlaps this OCR block."
        }

    if ocr_theme == "Other/Unknown" and speech_theme == "Other/Unknown":
        return {
            "agreement": "unknown_both",
            "final_theme": "Other/Unknown",
            "needs_review": True,
            "review_reason": "Both modalities are unknown."
        }

    if ocr_theme == "Other/Unknown" and speech_theme != "Other/Unknown":
        return {
            "agreement": "speech_only",
            "final_theme": speech_theme,
            "needs_review": True,
            "review_reason": "OCR theme is unknown; final theme relies on speech."
        }

    if speech_theme == "Other/Unknown" and ocr_theme != "Other/Unknown":
        return {
            "agreement": "ocr_only",
            "final_theme": ocr_theme,
            "needs_review": False,
            "review_reason": "Speech theme is unknown; OCR provides the usable theme."
        }

    if ocr_theme == speech_theme:
        return {
            "agreement": "high",
            "final_theme": ocr_theme,
            "needs_review": False,
            "review_reason": ""
        }

    if themes_are_related(ocr_theme, speech_theme):
        return {
            "agreement": "medium_related",
            "final_theme": ocr_theme,
            "needs_review": False,
            "review_reason": "Themes differ but belong to the same broad group."
        }

    return {
        "agreement": "low_conflict",
        "final_theme": ocr_theme,
        "needs_review": True,
        "review_reason": "OCR and speech themes conflict."
    }


def overlap_seconds(a_start, a_end, b_start, b_end):
    return max(0.0, min(float(a_end), float(b_end)) - max(float(a_start), float(b_start)))


## 2. Carregar outputs limpos de OCR e speech

Este notebook não importa nem executa os notebooks anteriores.  
Lê apenas os CSVs finais, para evitar trazer plots e diagnósticos que foram úteis no desenvolvimento, mas não são necessários no resultado final.


In [ ]:
ocr_blocks = load_required_csv(OCR_BLOCKS_FILE, "OCR consolidated blocks")
speech_windows = load_required_csv(SPEECH_WINDOWS_FILE, "Speech topic windows")

ocr_anchor = load_optional_csv(OCR_ANCHOR_FILE, "OCR anchor-to-piece candidates")
ocr_transition_candidates = load_optional_csv(OCR_TRANSITIONS_FILE, "OCR transition candidates")

speech_blocks = load_optional_csv(SPEECH_BLOCKS_FILE, "Speech consolidated blocks")
speech_transition_candidates = load_optional_csv(SPEECH_TRANSITIONS_FILE, "Speech transition candidates")

ocr_blocks = ensure_numeric_time_columns(ocr_blocks, "start_sec", "end_sec", "ocr_blocks")
speech_windows = ensure_numeric_time_columns(speech_windows, "start_sec", "end_sec", "speech_windows")

print("\nOCR blocks columns:")
print(list(ocr_blocks.columns))

print("\nSpeech windows columns:")
print(list(speech_windows.columns))

display(ocr_blocks.head())
display(speech_windows.head())


## 3. Normalizar nomes e preparar tabelas

Criamos versões com prefixos claros:

- `ocr_*` para informação vinda dos blocos OCR;
- `speech_*` para informação vinda das janelas speech.

A coluna de tempo em segundos é a chave comum.


In [ ]:
# ------------------------------------------------------------
# Preparar OCR blocks
# ------------------------------------------------------------
ocr_theme_col = first_existing_col(
    ocr_blocks,
    ["dominant_theme", "ocr_theme", "theme"],
    required=True,
    label="ocr_blocks"
)

ocr_text_col = first_existing_col(
    ocr_blocks,
    ["example_text", "ocr_text", "text", "title_text"],
    required=False,
    label="ocr_blocks"
)

ocr_blocks_norm = ocr_blocks.copy()

# Garantir ID limpo.
if "block_id" not in ocr_blocks_norm.columns:
    ocr_blocks_norm["block_id"] = np.arange(1, len(ocr_blocks_norm) + 1)

ocr_blocks_norm["block_id"] = pd.to_numeric(ocr_blocks_norm["block_id"], errors="coerce").astype("Int64")
ocr_blocks_norm = ocr_blocks_norm.sort_values("start_sec").reset_index(drop=True)

ocr_blocks_norm["ocr_theme"] = ocr_blocks_norm[ocr_theme_col].fillna("Other/Unknown").astype(str)
ocr_blocks_norm["ocr_text"] = (
    ocr_blocks_norm[ocr_text_col].fillna("").astype(str)
    if ocr_text_col is not None
    else ""
)

# Colunas opcionais importantes.
for col in ["themes_present", "candidates_present", "parties_present", "start_transition_reason", "micro_blocks"]:
    if col not in ocr_blocks_norm.columns:
        ocr_blocks_norm[col] = ""

ocr_blocks_norm = ocr_blocks_norm.rename(columns={
    "themes_present": "ocr_themes_present",
    "candidates_present": "ocr_candidates_present",
    "parties_present": "ocr_parties_present",
    "start_transition_reason": "ocr_start_transition_reason",
    "avg_confidence": "ocr_avg_confidence",
})

# ------------------------------------------------------------
# Preparar speech windows
# ------------------------------------------------------------
speech_theme_col = first_existing_col(
    speech_windows,
    ["dominant_theme", "speech_theme", "theme"],
    required=True,
    label="speech_windows"
)

speech_text_col = first_existing_col(
    speech_windows,
    ["text", "transcript", "clean_text", "example_text", "text_concat"],
    required=False,
    label="speech_windows"
)

speech_windows_norm = speech_windows.copy()

if "window_id" not in speech_windows_norm.columns:
    speech_windows_norm["window_id"] = np.arange(1, len(speech_windows_norm) + 1)

speech_windows_norm["window_id"] = pd.to_numeric(speech_windows_norm["window_id"], errors="coerce").astype("Int64")
speech_windows_norm = speech_windows_norm.sort_values("start_sec").reset_index(drop=True)

speech_windows_norm["speech_theme"] = speech_windows_norm[speech_theme_col].fillna("Other/Unknown").astype(str)
speech_windows_norm["speech_text"] = (
    speech_windows_norm[speech_text_col].fillna("").astype(str)
    if speech_text_col is not None
    else ""
)

for col in ["themes_present", "candidates_present", "parties_present", "source_segments", "transition_reason"]:
    if col not in speech_windows_norm.columns:
        speech_windows_norm[col] = ""

speech_windows_norm = speech_windows_norm.rename(columns={
    "themes_present": "speech_themes_present",
    "candidates_present": "speech_candidates_present",
    "parties_present": "speech_parties_present",
    "source_segments": "speech_source_segments",
    "transition_reason": "speech_transition_reason",
    "n_raw_words": "speech_n_raw_words",
    "n_words": "speech_n_words",
})

print("OCR blocks normalized:", ocr_blocks_norm.shape)
print("Speech windows normalized:", speech_windows_norm.shape)

display(ocr_blocks_norm[[
    "block_id", "start_time", "end_time", "duration_min", "ocr_theme", "ocr_themes_present", "ocr_text"
]].head(10))

display(speech_windows_norm[[
    "window_id", "start_time", "end_time", "duration_min", "speech_theme", "speech_themes_present", "speech_text"
]].head(10))


## 4. Alinhamento temporal OCR ↔ Speech

Para cada bloco OCR, procuramos todas as janelas speech que se sobrepõem no tempo.

Como o OCR tem blocos de duração variável e o speech tem janelas regulares, usamos **overlap temporal ponderado**.

Exemplo:

```text
OCR block: 09:27–13:03
Speech windows:
09:00–09:30 → overlap 3s
09:30–10:00 → overlap 30s
...
13:00–13:30 → overlap 3s
```


In [ ]:
alignment_rows = []

for _, ocr_row in ocr_blocks_norm.iterrows():
    ocr_start = float(ocr_row["start_sec"])
    ocr_end = float(ocr_row["end_sec"])
    ocr_duration = max(ocr_end - ocr_start, 1.0)

    candidates = speech_windows_norm[
        (speech_windows_norm["end_sec"] > ocr_start) &
        (speech_windows_norm["start_sec"] < ocr_end)
    ].copy()

    for _, sp_row in candidates.iterrows():
        sp_start = float(sp_row["start_sec"])
        sp_end = float(sp_row["end_sec"])
        sp_duration = max(sp_end - sp_start, 1.0)

        ov = overlap_seconds(ocr_start, ocr_end, sp_start, sp_end)

        if ov < MIN_OVERLAP_SEC:
            continue

        alignment_rows.append({
            "block_id": int(ocr_row["block_id"]),
            "ocr_start_sec": ocr_start,
            "ocr_end_sec": ocr_end,
            "ocr_start_time": seconds_to_hhmmss(ocr_start),
            "ocr_end_time": seconds_to_hhmmss(ocr_end),
            "ocr_duration_sec": ocr_duration,
            "ocr_duration_min": ocr_duration / 60,

            "window_id": int(sp_row["window_id"]),
            "speech_start_sec": sp_start,
            "speech_end_sec": sp_end,
            "speech_start_time": seconds_to_hhmmss(sp_start),
            "speech_end_time": seconds_to_hhmmss(sp_end),
            "speech_duration_sec": sp_duration,

            "overlap_sec": ov,
            "overlap_ratio_of_ocr_block": ov / ocr_duration,
            "overlap_ratio_of_speech_window": ov / sp_duration,

            "ocr_theme": ocr_row["ocr_theme"],
            "ocr_themes_present": ocr_row.get("ocr_themes_present", ""),
            "ocr_text": ocr_row.get("ocr_text", ""),

            "speech_theme": sp_row["speech_theme"],
            "speech_themes_present": sp_row.get("speech_themes_present", ""),
            "speech_candidates_present": sp_row.get("speech_candidates_present", ""),
            "speech_parties_present": sp_row.get("speech_parties_present", ""),
            "speech_text": sp_row.get("speech_text", ""),
            "speech_source_segments": sp_row.get("speech_source_segments", ""),
            "speech_transition_reason": sp_row.get("speech_transition_reason", ""),
        })

alignment_long = pd.DataFrame(alignment_rows)

print("Alignment rows:", len(alignment_long))
display(alignment_long.head(20))

alignment_long.to_csv(FUSION_OUTPUT_DIR / "fusion_ocr_speech_alignment_long.csv", index=False)


## 5. Agregar speech dentro de cada bloco OCR

Agora condensamos as janelas speech sobrepostas em uma linha por bloco OCR.

O tema speech dominante é escolhido pelo tempo total de sobreposição.  
Se houver temas `Other/Unknown` e temas conhecidos, damos preferência aos temas conhecidos.


In [ ]:
def dominant_speech_theme_for_group(group):
    if group.empty:
        return "No speech", 0.0, 0.0

    theme_overlap = (
        group.groupby("speech_theme")["overlap_sec"]
        .sum()
        .sort_values(ascending=False)
    )

    # Preferir temas conhecidos se existirem.
    known = theme_overlap[~theme_overlap.index.isin(["Other/Unknown", "No speech", ""])]
    if len(known) > 0:
        theme = known.index[0]
        support_sec = float(known.iloc[0])
    else:
        theme = theme_overlap.index[0]
        support_sec = float(theme_overlap.iloc[0])

    total_overlap = float(group["overlap_sec"].sum())
    support_ratio = support_sec / max(total_overlap, 1.0)

    return theme, support_sec, support_ratio


speech_by_ocr_rows = []

for _, ocr_row in ocr_blocks_norm.iterrows():
    block_id = int(ocr_row["block_id"])
    group = alignment_long[alignment_long["block_id"] == block_id].copy() if not alignment_long.empty else pd.DataFrame()

    if group.empty:
        speech_by_ocr_rows.append({
            "block_id": block_id,
            "speech_theme": "No speech",
            "speech_theme_support_sec": 0.0,
            "speech_theme_support_ratio": 0.0,
            "speech_total_overlap_sec": 0.0,
            "speech_coverage_ratio": 0.0,
            "n_speech_windows": 0,
            "speech_themes_present": "",
            "speech_candidates_present": "",
            "speech_parties_present": "",
            "speech_source_segments": "",
            "speech_text": "",
            "speech_excerpt": "",
        })
        continue

    speech_theme, support_sec, support_ratio = dominant_speech_theme_for_group(group)
    total_overlap = float(group["overlap_sec"].sum())
    ocr_duration = float(ocr_row["duration_sec"])

    speech_text = concat_texts(group.sort_values("speech_start_sec")["speech_text"].tolist(), max_chars=3000)
    speech_excerpt = speech_text[:700] + ("..." if len(speech_text) > 700 else "")

    speech_by_ocr_rows.append({
        "block_id": block_id,
        "speech_theme": speech_theme,
        "speech_theme_support_sec": support_sec,
        "speech_theme_support_ratio": support_ratio,
        "speech_total_overlap_sec": total_overlap,
        "speech_coverage_ratio": total_overlap / max(ocr_duration, 1.0),
        "n_speech_windows": int(group["window_id"].nunique()),
        "speech_themes_present": concat_unique(group["speech_themes_present"].tolist(), max_items=20),
        "speech_candidates_present": concat_unique(group["speech_candidates_present"].tolist(), max_items=20),
        "speech_parties_present": concat_unique(group["speech_parties_present"].tolist(), max_items=20),
        "speech_source_segments": concat_unique(group["speech_source_segments"].tolist(), max_items=80),
        "speech_text": speech_text,
        "speech_excerpt": speech_excerpt,
    })

speech_by_ocr = pd.DataFrame(speech_by_ocr_rows)

display(speech_by_ocr.head(20))
speech_by_ocr.to_csv(FUSION_OUTPUT_DIR / "fusion_speech_aggregated_by_ocr_block.csv", index=False)


## 6. Juntar OCR blocks + speech agregado

Esta tabela é a base do resultado final multimodal.


In [ ]:
fusion_blocks = ocr_blocks_norm.merge(
    speech_by_ocr,
    on="block_id",
    how="left"
)

# Preencher blocos sem speech, se existirem.
fusion_blocks["speech_theme"] = fusion_blocks["speech_theme"].fillna("No speech")
fusion_blocks["speech_coverage_ratio"] = fusion_blocks["speech_coverage_ratio"].fillna(0)
fusion_blocks["speech_excerpt"] = fusion_blocks["speech_excerpt"].fillna("")
fusion_blocks["speech_text"] = fusion_blocks["speech_text"].fillna("")

# Concordância OCR vs Speech
agreement_rows = []
for _, row in fusion_blocks.iterrows():
    agreement_rows.append(agreement_between_themes(row.get("ocr_theme"), row.get("speech_theme")))

agreement_df = pd.DataFrame(agreement_rows)

fusion_blocks = pd.concat([fusion_blocks.reset_index(drop=True), agreement_df.reset_index(drop=True)], axis=1)

# Se a cobertura speech é muito baixa, o bloco merece revisão mesmo que o tema pareça coerente.
fusion_blocks["low_speech_coverage"] = fusion_blocks["speech_coverage_ratio"] < MIN_SPEECH_COVERAGE_RATIO

fusion_blocks["needs_review"] = (
    fusion_blocks["needs_review"].astype(bool) |
    fusion_blocks["low_speech_coverage"].astype(bool)
)

fusion_blocks["review_reason"] = fusion_blocks.apply(
    lambda r: (
        (str(r["review_reason"]) + "; " if str(r["review_reason"]).strip() else "") +
        ("Low speech coverage inside OCR block." if r["low_speech_coverage"] else "")
    ).strip("; "),
    axis=1
)

display_cols = [
    "block_id", "start_time", "end_time", "duration_min",
    "ocr_theme", "speech_theme", "final_theme",
    "agreement", "speech_coverage_ratio", "needs_review",
    "ocr_text", "speech_excerpt"
]

display_cols = [c for c in display_cols if c in fusion_blocks.columns]

with pd.option_context("display.max_rows", 40, "display.max_columns", None, "display.max_colwidth", 220, "display.width", 2000):
    display(fusion_blocks[display_cols])


## 7. Integrar transições pivot/anchor → notícia detetadas pelo OCR

O OCR tem uma etapa específica para tentar detetar a passagem de pivot/anchor para peça/notícia.

O speech **não** tenta detetar esta transição diretamente.  
Por isso, no resultado final, esta informação entra como evidência OCR adicional.


In [ ]:
if ocr_anchor is not None and not ocr_anchor.empty:
    anchor_cols_keep = [
        "block_id",
        "candidate_status",
        "confidence",
        "pivot_to_piece_time",
        "pivot_to_piece_sec",
        "pivot_to_piece_frame",
        "candidate_offset_sec",
        "score",
        "persistence_windows",
        "evidence",
        "new_body_terms",
        "candidate_body_ocr",
    ]
    anchor_cols_keep = [c for c in anchor_cols_keep if c in ocr_anchor.columns]

    ocr_anchor_norm = ocr_anchor[anchor_cols_keep].copy()

    # Prefixar colunas, exceto block_id.
    rename_anchor = {
        c: f"anchor_{c}"
        for c in ocr_anchor_norm.columns
        if c != "block_id"
    }
    ocr_anchor_norm = ocr_anchor_norm.rename(columns=rename_anchor)

    fusion_blocks = fusion_blocks.merge(ocr_anchor_norm, on="block_id", how="left")

    fusion_blocks["anchor_candidate_status"] = fusion_blocks["anchor_candidate_status"].fillna("not_available")
    fusion_blocks["anchor_confidence"] = fusion_blocks["anchor_confidence"].fillna("none")

    fusion_blocks["anchor_to_piece_detected"] = fusion_blocks["anchor_candidate_status"].isin([
        "candidate_detected",
        "weak_candidate_no_persistence"
    ])

    print("Anchor-to-piece OCR information merged.")
    display(fusion_blocks[[
        "block_id", "start_time", "end_time",
        "ocr_theme", "speech_theme",
        "anchor_candidate_status", "anchor_confidence",
        "anchor_pivot_to_piece_time", "anchor_candidate_offset_sec",
        "anchor_evidence"
    ]].head(30))
else:
    fusion_blocks["anchor_candidate_status"] = "not_available"
    fusion_blocks["anchor_confidence"] = "none"
    fusion_blocks["anchor_to_piece_detected"] = False
    print("OCR anchor-to-piece file not available. Skipping this integration.")


## 8. Usar speech como validação adicional perto das fronteiras OCR

Esta parte é opcional e conservadora.

O speech não redefine as fronteiras OCR.  
Apenas verificamos se há uma mudança temática speech perto do início de cada bloco OCR.

Isto pode reforçar ou sinalizar dúvidas sobre uma fronteira.


In [ ]:
def nearest_speech_transition_to_boundary(block_start_sec, speech_transitions, tolerance_sec):
    if speech_transitions is None or speech_transitions.empty:
        return None

    st = speech_transitions.copy()

    if "start_sec" not in st.columns:
        return None

    st["start_sec"] = pd.to_numeric(st["start_sec"], errors="coerce")
    st = st.dropna(subset=["start_sec"]).copy()

    st["distance_to_boundary_sec"] = (st["start_sec"] - float(block_start_sec)).abs()
    near = st[st["distance_to_boundary_sec"] <= tolerance_sec].copy()

    if near.empty:
        return None

    return near.sort_values("distance_to_boundary_sec").iloc[0].to_dict()


speech_boundary_rows = []

for _, row in fusion_blocks.iterrows():
    block_start = float(row["start_sec"])
    nearest = nearest_speech_transition_to_boundary(
        block_start,
        speech_transition_candidates,
        SPEECH_TRANSITION_NEAR_BOUNDARY_SEC
    )

    if nearest is None:
        speech_boundary_rows.append({
            "block_id": int(row["block_id"]),
            "speech_transition_near_boundary": False,
            "nearest_speech_transition_time": "",
            "nearest_speech_transition_distance_sec": np.nan,
            "nearest_speech_transition_reason": "",
            "nearest_speech_transition_theme": "",
        })
    else:
        speech_boundary_rows.append({
            "block_id": int(row["block_id"]),
            "speech_transition_near_boundary": True,
            "nearest_speech_transition_time": nearest.get("start_time", seconds_to_hhmmss(nearest.get("start_sec"))),
            "nearest_speech_transition_distance_sec": nearest.get("distance_to_boundary_sec", np.nan),
            "nearest_speech_transition_reason": nearest.get("transition_reason", ""),
            "nearest_speech_transition_theme": nearest.get("dominant_theme", ""),
        })

speech_boundary_validation = pd.DataFrame(speech_boundary_rows)

fusion_blocks = fusion_blocks.merge(speech_boundary_validation, on="block_id", how="left")

display(fusion_blocks[[
    "block_id", "start_time", "end_time",
    "ocr_theme", "speech_theme", "agreement",
    "speech_transition_near_boundary",
    "nearest_speech_transition_time",
    "nearest_speech_transition_distance_sec",
    "nearest_speech_transition_reason"
]].head(30))


## 9. Tabela final multimodal

Esta é a tabela principal para a parte final da análise.

Interpretação recomendada:

```text
OCR theme / OCR text       → o que aparece como título/rodapé no ecrã.
Speech theme / excerpt     → o que foi falado dentro daquele bloco OCR.
Agreement                  → grau de concordância entre modalidades.
Needs review               → blocos onde convém validar manualmente.
Anchor-to-piece status     → evidência OCR da passagem pivot → peça/notícia.
```


In [ ]:
final_cols = [
    "block_id",
    "start_time",
    "end_time",
    "start_sec",
    "end_sec",
    "duration_min",

    "ocr_theme",
    "speech_theme",
    "final_theme",
    "agreement",
    "needs_review",
    "review_reason",

    "speech_coverage_ratio",
    "speech_theme_support_ratio",
    "n_speech_windows",

    "anchor_candidate_status",
    "anchor_confidence",
    "anchor_pivot_to_piece_time",
    "anchor_candidate_offset_sec",
    "anchor_score",
    "anchor_evidence",

    "speech_transition_near_boundary",
    "nearest_speech_transition_time",
    "nearest_speech_transition_distance_sec",
    "nearest_speech_transition_reason",

    "ocr_themes_present",
    "speech_themes_present",
    "ocr_candidates_present",
    "speech_candidates_present",
    "ocr_text",
    "speech_excerpt",
    "micro_blocks",
]

final_cols = [c for c in final_cols if c in fusion_blocks.columns]

fusion_final_blocks = fusion_blocks[final_cols].copy()

# Ordenar e arredondar algumas colunas para leitura.
fusion_final_blocks = fusion_final_blocks.sort_values("start_sec").reset_index(drop=True)

for col in ["duration_min", "speech_coverage_ratio", "speech_theme_support_ratio", "nearest_speech_transition_distance_sec"]:
    if col in fusion_final_blocks.columns:
        fusion_final_blocks[col] = pd.to_numeric(fusion_final_blocks[col], errors="coerce").round(3)

with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", 250, "display.width", 2200):
    display(fusion_final_blocks)

fusion_final_blocks.to_csv(FUSION_OUTPUT_DIR / "fusion_multimodal_blocks.csv", index=False)


## 10. Blocos que precisam de revisão

Estes são os casos mais úteis para discussão:

- OCR e speech discordam;
- falta speech suficiente dentro do bloco OCR;
- o OCR não conseguiu identificar tema mas o speech conseguiu;
- há evidência fraca/ambígua de transição pivot → notícia.


In [ ]:
review_blocks = fusion_final_blocks[fusion_final_blocks["needs_review"].astype(bool)].copy()

review_priority = []

for _, row in review_blocks.iterrows():
    reasons = str(row.get("review_reason", ""))

    if row.get("agreement") == "low_conflict":
        priority = "high"
    elif row.get("agreement") in ["no_speech", "unknown_both", "speech_only"]:
        priority = "medium"
    elif "Low speech coverage" in reasons:
        priority = "medium"
    else:
        priority = "low"

    review_priority.append(priority)

if not review_blocks.empty:
    review_blocks["review_priority"] = review_priority

review_cols = [
    "review_priority",
    "block_id", "start_time", "end_time", "duration_min",
    "ocr_theme", "speech_theme", "final_theme",
    "agreement", "review_reason",
    "anchor_candidate_status", "anchor_confidence",
    "ocr_text", "speech_excerpt"
]

review_cols = [c for c in review_cols if c in review_blocks.columns]

with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", 300, "display.width", 2200):
    display(review_blocks[review_cols] if not review_blocks.empty else review_blocks)

review_blocks.to_csv(FUSION_OUTPUT_DIR / "fusion_blocks_needing_review.csv", index=False)


## 11. Sínteses rápidas

Estas tabelas são pequenas e servem para apresentação/discussão.


In [ ]:
agreement_summary = (
    fusion_final_blocks["agreement"]
    .value_counts(dropna=False)
    .reset_index()
)
agreement_summary.columns = ["agreement", "n_blocks"]
agreement_summary["percentage"] = (agreement_summary["n_blocks"] / max(len(fusion_final_blocks), 1) * 100).round(2)

theme_summary = (
    fusion_final_blocks["final_theme"]
    .value_counts(dropna=False)
    .reset_index()
)
theme_summary.columns = ["final_theme", "n_blocks"]
theme_summary["percentage"] = (theme_summary["n_blocks"] / max(len(fusion_final_blocks), 1) * 100).round(2)

anchor_summary = (
    fusion_blocks["anchor_candidate_status"]
    .value_counts(dropna=False)
    .reset_index()
)
anchor_summary.columns = ["anchor_candidate_status", "n_blocks"]
anchor_summary["percentage"] = (anchor_summary["n_blocks"] / max(len(fusion_blocks), 1) * 100).round(2)

display(Markdown("### OCR vs Speech agreement"))
display(agreement_summary)

display(Markdown("### Final themes"))
display(theme_summary)

display(Markdown("### OCR anchor-to-piece status"))
display(anchor_summary)

agreement_summary.to_csv(FUSION_OUTPUT_DIR / "fusion_agreement_summary.csv", index=False)
theme_summary.to_csv(FUSION_OUTPUT_DIR / "fusion_theme_summary.csv", index=False)
anchor_summary.to_csv(FUSION_OUTPUT_DIR / "fusion_anchor_to_piece_summary.csv", index=False)


## 12. Visualizações finais mínimas

Mantemos apenas gráficos pequenos e interpretáveis.  
Os gráficos de diagnóstico detalhado ficam nos notebooks OCR e speech.


In [ ]:
if RUN_MINIMAL_PLOTS:
    plt.figure(figsize=(8, 4))
    plt.bar(agreement_summary["agreement"], agreement_summary["n_blocks"])
    plt.title("OCR vs Speech agreement by OCR block")
    plt.xlabel("agreement")
    plt.ylabel("n_blocks")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.bar(theme_summary["final_theme"].head(10), theme_summary["n_blocks"].head(10))
    plt.title("Top final multimodal themes")
    plt.xlabel("final_theme")
    plt.ylabel("n_blocks")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


## 13. Exemplos qualitativos

Selecionamos automaticamente alguns exemplos úteis:

- alta concordância;
- concordância média;
- conflito OCR vs speech;
- bloco com candidato pivot → peça/notícia.


In [ ]:
example_rows = []

def add_example(label, condition, max_rows=2):
    subset = fusion_final_blocks[condition].copy()
    if subset.empty:
        return
    subset = subset.sort_values(["start_sec"]).head(max_rows)
    subset["example_type"] = label
    example_rows.append(subset)

add_example("high_agreement", fusion_final_blocks["agreement"] == "high")
add_example("medium_related_agreement", fusion_final_blocks["agreement"] == "medium_related")
add_example("low_conflict", fusion_final_blocks["agreement"] == "low_conflict")
add_example("needs_review", fusion_final_blocks["needs_review"].astype(bool))

if "anchor_candidate_status" in fusion_final_blocks.columns:
    add_example("anchor_to_piece_candidate", fusion_final_blocks["anchor_candidate_status"].isin(["candidate_detected", "weak_candidate_no_persistence"]))

if example_rows:
    qualitative_examples = pd.concat(example_rows, ignore_index=True)
else:
    qualitative_examples = pd.DataFrame()

example_cols = [
    "example_type",
    "block_id", "start_time", "end_time", "duration_min",
    "ocr_theme", "speech_theme", "final_theme",
    "agreement", "needs_review", "review_reason",
    "anchor_candidate_status", "anchor_pivot_to_piece_time",
    "ocr_text", "speech_excerpt"
]

example_cols = [c for c in example_cols if c in qualitative_examples.columns]

with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", 350, "display.width", 2200):
    display(qualitative_examples[example_cols] if not qualitative_examples.empty else qualitative_examples)

qualitative_examples.to_csv(FUSION_OUTPUT_DIR / "fusion_qualitative_examples.csv", index=False)


## 14. Síntese automática para relatório/apresentação

Texto curto que pode ser adaptado para o relatório ou apresentação.


In [ ]:
n_blocks = len(fusion_final_blocks)
n_review = int(fusion_final_blocks["needs_review"].sum()) if "needs_review" in fusion_final_blocks else 0
n_high = int((fusion_final_blocks["agreement"] == "high").sum()) if "agreement" in fusion_final_blocks else 0
n_medium = int((fusion_final_blocks["agreement"] == "medium_related").sum()) if "agreement" in fusion_final_blocks else 0
n_low = int((fusion_final_blocks["agreement"] == "low_conflict").sum()) if "agreement" in fusion_final_blocks else 0

top_themes = (
    fusion_final_blocks["final_theme"]
    .value_counts()
    .head(5)
    .index
    .tolist()
)

summary_text = f'''
### Multimodal OCR + Speech Fusion Summary

The final newscast segmentation uses OCR as the temporal backbone because OCR captures visual lower-thirds, titles and anchor-to-piece transition candidates.
Speech is aligned with these OCR blocks using temporal overlap and is used to validate or enrich the topic assigned to each block.

- OCR blocks analysed: **{n_blocks}**
- Blocks with high OCR/speech agreement: **{n_high}**
- Blocks with related/medium agreement: **{n_medium}**
- Blocks with low conflict: **{n_low}**
- Blocks marked for manual review: **{n_review}**
- Main final themes: **{", ".join(top_themes)}**

Methodological interpretation:

- OCR defines the estimated news boundaries and duration.
- Speech validates whether the spoken content matches the OCR lower-third topic.
- When OCR and speech agree, confidence in the block topic increases.
- When they disagree, the block is flagged for manual review rather than automatically changing the OCR boundary.
- Anchor-to-piece transitions remain OCR-based evidence, because the speech pipeline does not directly detect visual anchor-to-piece scene changes.
'''

display(Markdown(summary_text))

with open(FUSION_OUTPUT_DIR / "fusion_draft_summary.md", "w", encoding="utf-8") as f:
    f.write(summary_text)


## 15. Outputs gerados

Ficheiros guardados em `outputs_fusion_ocr_speech/`:

```text
fusion_ocr_speech_alignment_long.csv
fusion_speech_aggregated_by_ocr_block.csv
fusion_multimodal_blocks.csv
fusion_blocks_needing_review.csv
fusion_agreement_summary.csv
fusion_theme_summary.csv
fusion_anchor_to_piece_summary.csv
fusion_qualitative_examples.csv
fusion_draft_summary.md
```

O ficheiro principal é:

```text
fusion_multimodal_blocks.csv
```

Este é o resultado final multimodal da parte OCR + Speech.
